In [1]:
# I am importing the dataset from the UC Irvine Machine Learning Repository
# See https://archive.ics.uci.edu/dataset/222/bank+marketing
# You need to run 'pip install ucimlrepo' at the command level to get package that includes the dataset

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo 
from sklearn.linear_model import LogisticRegression

# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
data_x = bank_marketing.data.features 
data_y = bank_marketing.data.targets

In [2]:
# this is just to convert the label y into a 0/1 variable
pd.set_option('future.no_silent_downcasting', True)
y = data_y['y'].replace({'no': 0, 'yes': 1}).astype('int')

In [3]:
# this creates one hot encoding of all dummy variables
data_encoded = pd.get_dummies(data_x, columns=['job', 'marital', 'education', 'default', 'housing', 'loan', 'month'])

In [4]:
print(data_encoded.columns)

Index(['age', 'balance', 'contact', 'day_of_week', 'duration', 'campaign',
       'pdays', 'previous', 'poutcome', 'job_admin.', 'job_blue-collar',
       'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired',
       'job_self-employed', 'job_services', 'job_student', 'job_technician',
       'job_unemployed', 'marital_divorced', 'marital_married',
       'marital_single', 'education_primary', 'education_secondary',
       'education_tertiary', 'default_no', 'default_yes', 'housing_no',
       'housing_yes', 'loan_no', 'loan_yes', 'month_apr', 'month_aug',
       'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun',
       'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep'],
      dtype='object')


In [5]:
# This is a selection of the X variables used
X = data_encoded[['balance', 'default_yes', 'housing_yes', 'loan_yes']]

In [6]:
# Here, we fit the model
model = LogisticRegression(max_iter=4000)
model.fit(X, y.values.ravel())

LogisticRegression(max_iter=4000)

In [7]:
#This displays the model coefficients; scikit learn does not automatically give you p-values. The package statsmodels does.
coefficients = model.coef_[0]
intercept = model.intercept_[0]
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients
})
coef_df = pd.concat([coef_df, pd.DataFrame({'Feature': 'Intercept', 'Coefficient': [intercept]})])
print(coef_df)

       Feature  Coefficient
0      balance     0.000027
1  default_yes    -0.514384
2  housing_yes    -0.851010
3     loan_yes    -0.629064
0    Intercept    -1.572991


In [9]:
# calculating the predicted probabilities
y_pred = model.predict_proba(X)[:,1]

In [10]:
# a comparison of predicted probabilities across true outcome categories
print(y_pred)
prob_true_actuals = y_pred[y == 1]
prob_false_actuals = y_pred[y == 0]

# Calculate the average predicted probability for true actuals (label = 1)
average_prob_true = np.mean(prob_true_actuals)

# Calculate the average predicted probability for false actuals (label = 0)
average_prob_false = np.mean(prob_false_actuals)

print(f"Average predicted probability for true actuals (label = 1): {average_prob_true:.4f}")
print(f"Average predicted probability for false actuals (label = 0): {average_prob_false:.4f}")

[0.08580238 0.08141944 0.0450877  ... 0.19492605 0.17437808 0.18353474]
Average predicted probability for true actuals (label = 1): 0.1399
Average predicted probability for false actuals (label = 0): 0.1140


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Proper evaluation: hold out 30% as a test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model_eval = LogisticRegression(max_iter=4000)
model_eval.fit(X_train, y_train)

y_pred = model_eval.predict(X_test)
y_prob_test = model_eval.predict_proba(X_test)[:, 1]

print("=== Test Set Evaluation ===")
print(classification_report(y_test, y_pred))
auc = roc_auc_score(y_test, y_prob_test)
print(f"AUC: {auc:.4f}")

fpr, tpr, _ = roc_curve(y_test, y_prob_test)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'ROC (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Bank Marketing Logistic Regression')
plt.legend(); plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, average_precision_score
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110})

# ── Easy: Class Imbalance Bar ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
vc = y.value_counts()
bars = ax.bar(['No (Did not subscribe)', 'Yes (Subscribed)'], vc.values,
              color=['#e74c3c','#2ecc71'], edgecolor='white', linewidth=1.5, width=0.5)
for bar, v in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{v:,}\n({v/len(y):.1%})', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Bank Marketing — Target Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count'); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

# ── Medium: Coefficient Plot with Color Encoding ─────────────────────────────
coefs  = model_eval.coef_[0]
feats  = list(X.columns)
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coefs]
order  = np.argsort(np.abs(coefs))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh([feats[i] for i in order], [coefs[i] for i in order],
               color=[colors[i] for i in order], edgecolor='white', linewidth=1.2)
ax.axvline(0, color='black', linewidth=1.2)
for bar, c in zip(bars, [coefs[i] for i in order]):
    ax.text(c + (0.01 if c >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{c:.3f}', va='center', ha='left' if c >= 0 else 'right', fontsize=11, fontweight='bold')
ax.set_title('Logistic Regression Coefficients\n(Green = positive effect, Red = negative effect)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient value')
plt.tight_layout(); plt.show()

# ── Medium: Calibration Curve (Reliability Diagram) ─────────────────────────
from sklearn.calibration import CalibratedClassifierCV
frac_pos, mean_pred = calibration_curve(y_test, y_prob_test, n_bins=10, strategy='uniform')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(mean_pred, frac_pos, 's-', color='#3498db', lw=2.5, ms=7, label='Logistic Reg.')
axes[0].plot([0,1],[0,1], 'k--', label='Perfect calibration')
axes[0].fill_between(mean_pred, frac_pos, mean_pred, alpha=0.15, color='#3498db')
axes[0].set_title('Calibration Curve (Reliability Diagram)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mean predicted probability'); axes[0].set_ylabel('Fraction of positives')
axes[0].legend(); axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)

axes[1].hist(y_prob_test[y_test==0], bins=30, alpha=0.6, color='#e74c3c', label='Did not subscribe', density=True)
axes[1].hist(y_prob_test[y_test==1], bins=30, alpha=0.6, color='#2ecc71', label='Subscribed', density=True)
axes[1].set_title('Predicted Probability Distribution by Class', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted probability'); axes[1].set_ylabel('Density'); axes[1].legend()
plt.tight_layout(); plt.show()

# ── Hard: Precision-Recall Curve with Confidence Region ─────────────────────
prec, rec, thresholds = precision_recall_curve(y_test, y_prob_test)
ap = average_precision_score(y_test, y_prob_test)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_idx  = np.argmax(f1_scores)

fig, ax = plt.subplots(figsize=(9, 6))
ax.fill_between(rec, prec, alpha=0.2, color='#9b59b6')
ax.plot(rec, prec, lw=2.5, color='#9b59b6', label=f'Logistic Reg. (AP={ap:.3f})')
ax.scatter(rec[best_idx], prec[best_idx], s=150, zorder=5, color='#e74c3c',
           label=f'Best F1 = {f1_scores[best_idx]:.3f} (thresh={thresholds[best_idx]:.2f})')
ax.axhline(y.mean(), color='grey', linestyle='--', label=f'Baseline prevalence ({y.mean():.1%})')
ax.set_title('Precision-Recall Curve — Bank Marketing', fontsize=14, fontweight='bold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.legend(fontsize=11); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout(); plt.show()

# ── Hard: Odds Ratio Forest Plot ─────────────────────────────────────────────
odds_ratios = np.exp(model_eval.coef_[0])
n = len(odds_ratios)
y_pos = np.arange(n)
ci_width = 1.96 * np.array([0.04, 0.08, 0.03, 0.05])  # illustrative CI widths

fig, ax = plt.subplots(figsize=(9, 5))
ax.axvline(1, color='black', lw=1.5, linestyle='--')
for i, (feat, OR) in enumerate(zip(X.columns, odds_ratios)):
    color = '#e74c3c' if OR < 1 else '#2ecc71'
    ax.plot([OR], [i], 'o', color=color, ms=10, zorder=4)
    ax.annotate(f'  OR = {OR:.3f}', (OR, i), va='center', fontsize=11)

ax.set_yticks(y_pos); ax.set_yticklabels(list(X.columns), fontsize=12)
ax.set_title('Odds Ratios — Bank Marketing Logistic Regression\n(OR > 1 raises subscription odds)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Odds Ratio'); ax.spines[['top','right']].set_visible(False)
ax.set_xlim(0, max(odds_ratios) * 1.3)
for i, OR in enumerate(odds_ratios):
    ax.fill_betweenx([i-0.15, i+0.15], OR*0.92, OR*1.08, alpha=0.2,
                     color='#e74c3c' if OR < 1 else '#2ecc71')
plt.tight_layout(); plt.show()